In [ ]:
import numpy as np
from environments.simple_trading_env import SimpleTradingEnv
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from stable_baselines3 import PPO
import pandas as pd
import torch
import matplotlib.pyplot as plt

# === CONFIGURATION ===
DATA_SYMBOL = 'BTCUSDT'
DATA_TIMEFRAME = '5m'
DATA_PATH = f'data/binance-{DATA_SYMBOL}-{DATA_TIMEFRAME}.pkl'
MODEL_PATH = "trading_bot"
LOOKBACK_WINDOW = 288

# Load data
df = pd.read_pickle(DATA_PATH)
print(f'✓ Loaded {len(df):,} rows for {DATA_SYMBOL} {DATA_TIMEFRAME}')

# Test on unseen data
total_timesteps = 2000
test_start = 15_536
test_data = df.iloc[test_start:test_start + total_timesteps].reset_index(drop=True)
print(f'✓ Testing on rows {test_start:,} to {test_start + total_timesteps:,}')

# Create test environment
test_env = SimpleTradingEnv(test_data, device="cuda", lookback_window=LOOKBACK_WINDOW)
test_env = Monitor(test_env)
test_env = DummyVecEnv([lambda: test_env])

# Load trained model
model = PPO.load(MODEL_PATH, env=test_env, device="cuda")
print(f'✓ Loaded model from {MODEL_PATH}\n')

# === FEATURE ACTIVATION TRACKING ===
extractor = model.policy.features_extractor
HOOKABLE_PATTERNS = ['_cnn', '_output', '_encoder', '_transformer']

# Auto-discover all hookable modules
available_features = {}
for attr_name in dir(extractor):
    if attr_name.startswith('_'):
        continue
    if any(pattern in attr_name for pattern in HOOKABLE_PATTERNS):
        attr = getattr(extractor, attr_name)
        if isinstance(attr, torch.nn.Module):
            display_name = attr_name.replace('_', ' ').title().replace(' ', '_')
            available_features[attr_name] = display_name

# Initialize tracking
feature_activations = {display_name: [] for display_name in available_features.values()}
activations_storage = []

def make_hook(name):
    def hook(module, input, output):
        if isinstance(output, tuple):
            output = output[0]
        if isinstance(output, torch.Tensor):
            activations_storage.append({
                'name': name,
                'magnitude': output.abs().mean().item()
            })
    return hook

# Register hooks
hooks = []
for module_name, display_name in available_features.items():
    module = getattr(extractor, module_name)
    hook = module.register_forward_hook(make_hook(display_name))
    hooks.append(hook)

# Run evaluation
print("Running evaluation...")
obs = test_env.reset()
total_reward = 0
reward_history = []
steps = 0
last_env = []

while True:
    activations_storage.clear()
    action, _states = model.predict(obs, deterministic=True)
    
    for act in activations_storage:
        feature_activations[act['name']].append(act['magnitude'])
    
    obs, reward, done, info = test_env.step(action)
    total_reward += reward[0]
    reward_history.append(reward[0])
    steps += 1
    if done[0]:
        break
    last_env.append(test_env.envs[0].env.history[-1])

# Clean up hooks
for hook in hooks:
    hook.remove()

# Calculate average activations
avg_activations = {}
for name, values in feature_activations.items():
    if len(values) > 0:
        avg_activations[name] = np.mean(values)

# === SIMPLE REPORT ===
print("\n" + "="*80)
print("EVALUATION REPORT")
print("="*80)

# Reward stats
print(f"\n📊 REWARD STATISTICS:")
print(f"   Total Reward:    {total_reward:+10.2f}")
print(f"   Min Reward:      {min(reward_history):+10.2f}")
print(f"   Max Reward:      {max(reward_history):+10.2f}")
print(f"   Avg Reward/Step: {total_reward/steps:+10.4f}")
print(f"   Total Steps:     {steps:10}")

# PnL stats
if len(last_env):
    trades = last_env[-1].get('trades', [])
    if len(trades) > 0:
        total_pnl = sum(t.get('pnl', 0) for t in trades)
        tp_count = sum(1 for t in trades if 'TP' in t.get('reason', ''))
        sl_count = sum(1 for t in trades if 'SL' in t.get('reason', ''))
        
        print(f"\n💰 TRADING STATISTICS:")
        print(f"   Total Trades:    {len(trades):10}")
        print(f"   Total PnL:       {total_pnl:+10.2f}")
        print(f"   Avg PnL/Trade:   {total_pnl/len(trades):+10.2f}")
        print(f"   TP Exits:        {tp_count:10} ({tp_count/len(trades)*100:5.1f}%)")
        print(f"   SL Exits:        {sl_count:10} ({sl_count/len(trades)*100:5.1f}%)")
    else:
        print(f"\n⚠️  NO TRADES EXECUTED")

# === FEATURE USAGE GRAPH ===
if avg_activations:
    print(f"\n📈 FEATURE ACTIVATIONS ({len(avg_activations)} groups):")
    
    # Sort by magnitude
    sorted_features = sorted(avg_activations.items(), key=lambda x: x[1], reverse=True)
    names, values = zip(*sorted_features)
    
    # Create bar chart
    plt.figure(figsize=(14, 6))
    colors = plt.cm.viridis(np.linspace(0, 1, len(names)))
    plt.bar(range(len(names)), values, color=colors)
    plt.xlabel('Feature Group', fontsize=12)
    plt.ylabel('Average Activation Magnitude', fontsize=12)
    plt.title('Feature Group Usage During Inference', fontsize=14, fontweight='bold')
    plt.xticks(range(len(names)), names, rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Print top 10 values
    print("\n   Top 10 Most Active Features:")
    for i, (name, value) in enumerate(sorted_features[:10], 1):
        print(f"   {i:2}. {name:30}: {value:.4f}")
else:
    print("\n⚠️  No feature activations recorded")

print("\n" + "="*80)

In [ ]:
# === TRADES REPORT ===
if len(last_env):
    trades = last_env[-1].get('trades', [])
    
    if len(trades) > 0:
        # Action name mapping
        action_names = {0: 'HOLD', 1: 'LONG', 2: 'SHORT', 3: 'CLOSE'}
        
        # Build a mapping of step_open to action for each trade
        trade_actions = {}
        for env_state in last_env:
            step = env_state.get('step', 0)
            action = env_state.get('action', [0, 0, 0])
            # If action is opening a position (1=LONG, 2=SHORT)
            if int(action[0]) in [1, 2]:
                trade_actions[step] = action
        
        # Create DataFrame from trades
        trades_list = []
        for t in trades:
            step_open = t.get('step_open', 0)
            action = trade_actions.get(step_open)  # Default if not found
            
            trades_list.append({
                'PnL': t.get('pnl', 0),
                'PnL%': t.get('pnl_percent', 0) * 100,
                'Duration': t.get('duration', 0),
                'Exit': t.get('reason', 'N/A'),
                'Entry Price': t.get('entry_price', 0),
                'Exit Price': t.get('exit_price', 0),
                'Dir': action_names.get(int(action[0]), '?'),
                'RR': int(action[1]),
                'ATR': int(action[2])
            })
        
        trades_df = pd.DataFrame(trades_list)
        
        # Format numeric columns
        pd.options.display.float_format = '{:,.2f}'.format
        
        print("="*80)
        print(f"ALL TRADES ({len(trades)} total)")
        print("="*80)
        print("Columns: Dir=Direction, RR=Risk-Reward (0-9 → 1-10x), ATR=ATR Multiplier (0-9 → 2.0-4.7x)")
        print("="*80)
        
        # Display with pandas styling
        display(trades_df.style.format({
            'PnL': '{:+,.2f}',
            'PnL%': '{:+.2f}%',
            'Duration': '{:.0f}',
            'Entry Price': '{:,.2f}',
            'Exit Price': '{:,.2f}',
            'RR': '{:.0f}',
            'ATR': '{:.0f}'
        }).background_gradient(subset=['PnL'], cmap='RdYlGn', vmin=-trades_df['PnL'].abs().max(), vmax=trades_df['PnL'].abs().max()))
        
        # Summary stats
        print("\n" + "="*80)
        print("TRADE STATISTICS")
        print("="*80)
        
        stats_df = pd.DataFrame({
            'Metric': ['Total Trades', 'Winning Trades', 'Losing Trades', 'Win Rate', 
                       'Total PnL', 'Avg PnL', 'Best Trade', 'Worst Trade', 'Avg Duration',
                       'Avg RR Ratio', 'Avg ATR Mult'],
            'Value': [
                len(trades),
                len(trades_df[trades_df['PnL'] > 0]),
                len(trades_df[trades_df['PnL'] < 0]),
                f"{len(trades_df[trades_df['PnL'] > 0])/len(trades)*100:.1f}%",
                f"{trades_df['PnL'].sum():+,.2f}",
                f"{trades_df['PnL'].mean():+,.2f}",
                f"{trades_df['PnL'].max():+,.2f}",
                f"{trades_df['PnL'].min():+,.2f}",
                f"{trades_df['Duration'].mean():.1f} steps",
                f"{trades_df['RR'].mean():.1f} (= {1.0 + trades_df['RR'].mean():.1f}x)",
                f"{trades_df['ATR'].mean():.1f} (= {2.0 + trades_df['ATR'].mean() * 0.3:.2f}x)"
            ]
        })
        
        display(stats_df)
        
        # Action parameter analysis
        print("\n" + "="*80)
        print("ACTION PARAMETER ANALYSIS")
        print("="*80)
        
        print(f"\nRisk-Reward Ratio Distribution:")
        print(trades_df['RR'].value_counts().sort_index())
        
        print(f"\nATR Multiplier Distribution:")
        print(trades_df['ATR'].value_counts().sort_index())
        
        # Winning vs Losing action preferences
        winning_trades = trades_df[trades_df['PnL'] > 0]
        losing_trades = trades_df[trades_df['PnL'] < 0]
        
        if len(winning_trades) > 0 and len(losing_trades) > 0:
            print(f"\n📊 Average Parameters:")
            print(f"   Winning Trades: RR={winning_trades['RR'].mean():.1f}, ATR={winning_trades['ATR'].mean():.1f}")
            print(f"   Losing Trades:  RR={losing_trades['RR'].mean():.1f}, ATR={losing_trades['ATR'].mean():.1f}")
    else:
        print("⚠️  NO TRADES EXECUTED")
else:
    print("⚠️  No environment history available")